# Confusion Matrices, ROC Curves & Prediction Cluster Analysis — 154-Class Models
**Dissertation: Real-Time ASL Recognition** | Owasu Brown | NU 2025–2026

## What this notebook produces (per model)
| Output | Saved to `results/end_point/` |
|---|---|
| Top-25 confused pairs table | `{model}_confusion_table.png` + `_confusion_pairs.csv` |
| Macro-averaged ROC curve | `{model}_roc_macro.png` |
| Per-class ROC (top 10 by AUC) | `{model}_roc_per_class.png` |
| **Top-1 prediction cluster map** | `{model}_cluster_top1.png` + `_cluster_top1_summary.csv` |
| **Top-5 prediction cluster map** | `{model}_cluster_top5.png` + `_cluster_top5_summary.csv` |
| **Top-20 prediction cluster map** | `{model}_cluster_top20.png` + `_cluster_top20_summary.csv` |

All outputs overwrite existing files. Run order: Cell 1 → 2 → 3 → 4 → 5.

## Data sources
| Model | Proba array | y_true source |
|---|---|---|
| InceptionTime | `inc_154/inc_154_proba_test.npy` | `data/npz/ASL_154_test_cif.npz` key `y` |
| Random Forest | `rf_154/test/rf_154_proba_test.npy` | `data/csv/ASL_154_test_rf.csv` |
| Logistic Regression | `lr_154/lr_154_proba_test.npy` | `data/csv/ASL_154_test_rf.csv` |
| CIF (50 trees) | `cif_154/cif_154_proba_test.npy` | `data/npz/ASL_154_test_cif.npz` key `y` |

Labels loaded from `data/meta/ASL_154_label_encoder.pkl`.


In [1]:
# ── Cell 1: Install packages ──────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib',
    'seaborn', 'joblib', 'umap-learn', '-q'
])
print('✅ Packages ready')


✅ Packages ready


In [2]:
# ── Cell 2: Mount Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
# ── Cell 3: Paths & label encoder ────────────────────────────────────────
import os, sys, json, warnings
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings('ignore')

PROJECT_DIR = ('/content/drive/MyDrive/Consultant/Colab_Notebooks/'
               'Obrown_Dissertation_NU_25/OBrown_DIS9300_v2')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
END_DIR     = os.path.join(RESULTS_DIR, 'end_point')
os.makedirs(END_DIR, exist_ok=True)

# ── Label encoder — search common locations ───────────────────────────────
LE_CANDIDATES = [
    os.path.join(PROJECT_DIR, 'data', 'meta', 'ASL_154_label_encoder.pkl'),
    os.path.join(PROJECT_DIR, 'ASL_154_label_encoder.pkl'),
    os.path.join(PROJECT_DIR, 'data', 'ASL_154_label_encoder.pkl'),
]
LE_PATH = next((p for p in LE_CANDIDATES if os.path.exists(p)), None)
if LE_PATH is None:
    raise FileNotFoundError(
        'ASL_154_label_encoder.pkl not found. Searched:\n  ' +
        '\n  '.join(LE_CANDIDATES)
    )

le          = joblib.load(LE_PATH)
LABEL_NAMES = list(le.classes_)
N_CLASSES   = len(LABEL_NAMES)

print(f'✅ Label encoder : {N_CLASSES} classes  ({LE_PATH})')
print(f'   First 5 : {LABEL_NAMES[:5]}')
print(f'   Last  5 : {LABEL_NAMES[-5:]}')
print(f'   Output  : {END_DIR}')


✅ Label encoder : 154 classes  (/content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/data/meta/ASL_154_label_encoder.pkl)
   First 5 : [np.str_('accident'), np.str_('africa'), np.str_('apple'), np.str_('argue'), np.str_('baby')]
   Last  5 : [np.str_('wrong'), np.str_('year'), np.str_('yes'), np.str_('yesterday'), np.str_('you')]
   Output  : /content/drive/MyDrive/Consultant/Colab_Notebooks/Obrown_Dissertation_NU_25/OBrown_DIS9300_v2/results/end_point


In [8]:
# ── Cell 4: Core functions ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import seaborn as sns

_BLUE = '#2E75B6'
_RED  = '#E63946'
_DARK = '#1F3864'
_GREY = '#6C757D'


# ══════════════════════════════════════════════════════════════════════════
# 1.  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════

def load_proba_and_labels(slot):
    """
    Load y_true (int), y_pred (int), y_proba (float32) for a model slot.
    slot keys: npy_path, y_true_src ('npz'|'csv'), y_src_path
    """
    npy_path   = slot['npy_path']
    y_true_src = slot['y_true_src']
    y_src_path = slot['y_src_path']

    if not os.path.exists(npy_path):
        raise FileNotFoundError(f'Proba array not found: {npy_path}')
    if not os.path.exists(y_src_path):
        raise FileNotFoundError(f'y_true source not found: {y_src_path}')

    y_proba = np.load(npy_path, allow_pickle=True).astype(np.float32)

    if y_true_src == 'npz':
        d = np.load(y_src_path, allow_pickle=True)
        for key in ['y', 'y_test', 'labels', 'label']:
            if key in d:
                y_true = d[key].astype(int)
                break
        else:
            raise KeyError(f'No label key in {y_src_path}. Keys: {list(d.keys())}')
    else:   # csv
        df  = pd.read_csv(y_src_path)
        col = next((c for c in df.columns
                    if c.lower() in ['label', 'gloss', 'y_true', 'class']), None)
        if col is None:
            raise KeyError(f'No label column in {y_src_path}. Cols: {list(df.columns)}')
        raw    = df[col].values
        lmap   = {v: i for i, v in enumerate(np.unique(raw))}
        y_true = np.array([lmap[v] for v in raw])

    n       = min(len(y_true), len(y_proba))
    y_true  = y_true[:n]
    y_proba = y_proba[:n]
    y_pred  = np.argmax(y_proba, axis=1)
    return y_true, y_pred, y_proba


# ══════════════════════════════════════════════════════════════════════════
# 2.  CONFUSION TABLE
# ══════════════════════════════════════════════════════════════════════════

def plot_confusion_table(y_true, y_pred, label_names, model_name, top_n=25):
    n_cls = len(label_names)
    cm    = confusion_matrix(y_true, y_pred, labels=list(range(n_cls)))

    rows_list = [
        {'True Sign': label_names[i], 'Predicted As': label_names[j],
         'Count': int(cm[i, j])}
        for i in range(n_cls) for j in range(n_cls)
        if i != j and cm[i, j] > 0
    ]
    if not rows_list:
        print(f'  ⚠️  No off-diagonal errors for {model_name}'); return None

    error_df = (pd.DataFrame(rows_list)
                .sort_values('Count', ascending=False)
                .head(top_n).reset_index(drop=True))
    error_df.index += 1

    csv_path = os.path.join(END_DIR, f'{model_name}_confusion_pairs.csv')
    error_df.to_csv(csv_path)
    print(f'  ✅ Confusion CSV       : {os.path.basename(csv_path)}')

    fig_h = max(6, 0.42 * len(error_df) + 2.5)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis('off')

    col_labels = ['Rank', 'True Sign', 'Predicted As', 'Count']
    table_data = [[str(i), r['True Sign'], r['Predicted As'], str(r['Count'])]
                  for i, r in error_df.iterrows()]

    tbl = ax.table(cellText=table_data, colLabels=col_labels,
                   cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(9)
    q75 = error_df['Count'].quantile(0.75)

    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor(_DARK)
            cell.set_text_props(color='white', fontweight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#EEF2F7')
        else:
            cell.set_facecolor('#FFFFFF')
        cell.set_edgecolor('#CCCCCC')
        if row > 0 and int(table_data[row-1][3]) >= q75:
            cell.set_edgecolor(_RED)

    total_err = error_df['Count'].sum()
    total_all = int(cm.sum() - np.diag(cm).sum())
    ax.set_title(
        f'Top {top_n} Most Confused Class Pairs  —  {model_name}\n'
        f'{total_err} of {total_all} total misclassifications '
        f'({100*total_err/max(total_all,1):.1f}%)',
        fontsize=10, color=_DARK, pad=12)
    plt.tight_layout()
    png_path = os.path.join(END_DIR, f'{model_name}_confusion_table.png')
    plt.savefig(png_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  ✅ Confusion table PNG : {os.path.basename(png_path)}')


# ══════════════════════════════════════════════════════════════════════════
# 3.  ROC CURVES
# ══════════════════════════════════════════════════════════════════════════

def plot_roc_curves(y_true, y_proba, label_names, model_name, top_n_classes=10):
    n_cls = len(label_names)
    y_bin = label_binarize(y_true, classes=list(range(n_cls)))

    all_fpr  = np.linspace(0, 1, 500)
    mean_tpr = np.zeros(500)
    auc_per_class = {}

    for i in range(n_cls):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        a = auc(fpr, tpr)
        auc_per_class[i] = a
        mean_tpr += np.interp(all_fpr, fpr, tpr)

    n_valid = len(auc_per_class)
    if n_valid == 0:
        print(f'  ⚠️  No valid ROC curves for {model_name}'); return

    mean_tpr /= n_valid
    macro_auc = float(np.mean(list(auc_per_class.values())))

    # Macro ROC
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot(all_fpr, mean_tpr, color=_BLUE, linewidth=2.5,
            label=f'Macro-average ROC  (AUC = {macro_auc:.3f})')
    ax.fill_between(all_fpr, mean_tpr, alpha=0.12, color=_BLUE)
    ax.plot([0,1],[0,1], color=_GREY, linestyle='--',
            linewidth=1, label='Random classifier (AUC = 0.500)')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate',  fontsize=12)
    ax.set_title(f'Macro-Averaged ROC — {model_name}\n'
                 f'{n_valid} classes  |  Macro AUC = {macro_auc:.3f}',
                 fontsize=11, color=_DARK)
    ax.legend(fontsize=10); ax.set_xlim(0,1); ax.set_ylim(0,1.02); ax.grid(alpha=0.3)
    plt.tight_layout()
    macro_path = os.path.join(END_DIR, f'{model_name}_roc_macro.png')
    plt.savefig(macro_path, dpi=150); plt.close()
    print(f'  ✅ Macro ROC PNG       : {os.path.basename(macro_path)}  (AUC={macro_auc:.4f})')

    # Per-class ROC top-N by AUC
    top_idx = sorted(auc_per_class, key=auc_per_class.get, reverse=True)[:top_n_classes]
    cmap    = plt.cm.get_cmap('tab10', len(top_idx))
    fig, ax = plt.subplots(figsize=(10, 8))
    for k, cls_idx in enumerate(top_idx):
        fpr, tpr, _ = roc_curve(y_bin[:, cls_idx], y_proba[:, cls_idx])
        ax.plot(fpr, tpr, color=cmap(k), linewidth=1.8,
                label=f'{label_names[cls_idx]}  (AUC={auc_per_class[cls_idx]:.3f})')
    ax.plot([0,1],[0,1], color=_GREY, linestyle='--', linewidth=1, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate',  fontsize=12)
    ax.set_title(f'Per-Class ROC — Top {top_n_classes} by AUC\n{model_name}',
                 fontsize=11, color=_DARK)
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlim(0,1); ax.set_ylim(0,1.02); ax.grid(alpha=0.3)
    plt.tight_layout()
    pc_path = os.path.join(END_DIR, f'{model_name}_roc_per_class.png')
    plt.savefig(pc_path, dpi=150); plt.close()
    print(f'  ✅ Per-class ROC PNG   : {os.path.basename(pc_path)}')


# ══════════════════════════════════════════════════════════════════════════
# 4.  PREDICTION CLUSTER ANALYSIS  (Top-1, Top-5, Top-20)
# ══════════════════════════════════════════════════════════════════════════
#
# For a given top-K:
#   • Select the K classes with highest mean predicted probability
#     across the test set (i.e. the classes the model "talks about" most).
#   • Extract the n_samples × K probability sub-matrix for those classes.
#   • Reduce to 2-D with PCA (fast, reproducible) then optionally t-SNE.
#   • K-Means cluster the 2-D embedding; number of clusters = K (capped at 8).
#   • Colour points by TRUE class; mark cluster centroids with ★.
#   • Annotate each centroid with the dominant true class name.
#   • Draw overlap ellipses (1-σ) around each cluster to show confusion zones.
#   • Save PNG + summary CSV to END_DIR.
#
# "Overlap" = two cluster ellipses intersect → those class groups are
#  confused with each other by the model.

def _ellipse_params(points):
    """Return (cx, cy, w, h, angle_deg) for a 1-sigma ellipse around points."""
    if len(points) < 3:
        cx, cy = points.mean(axis=0)
        return cx, cy, 0.05, 0.05, 0
    cx, cy = points.mean(axis=0)
    cov    = np.cov(points.T)
    vals, vecs = np.linalg.eigh(cov)
    vals   = np.sqrt(np.abs(vals))          # std dev along each axis
    angle  = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
    w, h   = 2 * vals[-1], 2 * vals[0]     # 1-sigma axes
    return cx, cy, w, h, angle


def _ellipses_overlap(e1, e2):
    """
    Rough overlap test: do the bounding boxes of the two ellipses intersect?
    Good enough for visualisation annotation.
    """
    cx1, cy1, w1, h1, _ = e1
    cx2, cy2, w2, h2, _ = e2
    return (abs(cx1 - cx2) < (w1 + w2) / 2 and
            abs(cy1 - cy2) < (h1 + h2) / 2)


def plot_cluster_analysis(y_true, y_proba, label_names, model_name, top_k):
    """
    Cluster the probability space for the top-K predicted classes.
    Produces one PNG and one CSV in END_DIR.

    Parameters
    ----------
    top_k : int — 1, 5, or 20 (any value works)
    """
    n_cls   = len(label_names)
    n_samp  = len(y_proba)
    tag     = f'top{top_k}'

    # ── Step 1: identify top-K classes by mean predicted probability ──────
    mean_proba = y_proba.mean(axis=0)           # (n_classes,)
    top_k_idx  = np.argsort(mean_proba)[-top_k:][::-1]   # descending
    top_k_names = [label_names[i] for i in top_k_idx]

    # ── Step 2: sub-matrix for those classes ─────────────────────────────
    X_sub = y_proba[:, top_k_idx]               # (n_samples, top_k)

    # ── Step 3: 2-D reduction ─────────────────────────────────────────────
    n_components = min(2, X_sub.shape[1])
    if X_sub.shape[1] >= 2:
        pca  = PCA(n_components=2, random_state=42)
        X_2d = pca.fit_transform(X_sub)
        var_explained = pca.explained_variance_ratio_.sum() * 100
        reduction_label = f'PCA  ({var_explained:.1f}% variance explained)'
    else:
        # top_k == 1: just use the raw probability on x-axis with jitter
        X_2d = np.column_stack([X_sub[:, 0],
                                 np.random.default_rng(42).uniform(
                                     -0.01, 0.01, n_samp)])
        reduction_label = 'Probability score (jittered y)'

    # ── Step 4: K-Means clustering ────────────────────────────────────────
    n_clusters = min(top_k, 8, n_samp)          # cap at 8 for readability
    if n_clusters < 2:
        n_clusters = 2
    km       = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = km.fit_predict(X_2d)

    # Silhouette score
    try:
        sil = silhouette_score(X_2d, clusters) if n_clusters > 1 else float('nan')
    except Exception:
        sil = float('nan')

    # ── Step 5: Build cluster summary ─────────────────────────────────────
    cluster_rows = []
    for c in range(n_clusters):
        mask        = clusters == c
        true_in_c   = y_true[mask]
        dom_idx     = np.bincount(true_in_c, minlength=n_cls).argmax()
        dom_name    = label_names[dom_idx]
        dom_pct     = (true_in_c == dom_idx).mean() * 100
        # Which top-K classes are represented in this cluster?
        pred_in_c   = np.argmax(y_proba[mask], axis=1)
        topk_in_c   = [label_names[i] for i in top_k_idx
                       if (pred_in_c == i).any()]
        cluster_rows.append({
            'Cluster'          : c,
            'n_samples'        : int(mask.sum()),
            'Dominant_true'    : dom_name,
            'Dominant_pct'     : round(dom_pct, 1),
            'Top_k_classes_present': ', '.join(topk_in_c),
        })

    summary_df = pd.DataFrame(cluster_rows)
    csv_path   = os.path.join(END_DIR, f'{model_name}_cluster_{tag}_summary.csv')
    summary_df.to_csv(csv_path, index=False)

    # ── Step 6: Overlap detection ─────────────────────────────────────────
    ellipses = {}
    for c in range(n_clusters):
        pts = X_2d[clusters == c]
        ellipses[c] = _ellipse_params(pts)

    overlapping_pairs = []
    for a in range(n_clusters):
        for b in range(a+1, n_clusters):
            if _ellipses_overlap(ellipses[a], ellipses[b]):
                overlapping_pairs.append((a, b))

    # ── Step 7: Plot ──────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 9))

    # Colour points by TRUE class (only top-K true classes get distinct colours;
    # all others are plotted in light grey so the eye is drawn to the focal classes)
    true_in_topk = np.isin(y_true, top_k_idx)
    cmap_pts = plt.cm.get_cmap('tab20', top_k)

    # Grey background: samples whose true class is NOT in top-K
    ax.scatter(X_2d[~true_in_topk, 0], X_2d[~true_in_topk, 1],
               c='#DDDDDD', s=6, alpha=0.3, zorder=1, label='Other true class')

    # Coloured foreground: samples whose true class IS in top-K
    topk_set   = {idx: k for k, idx in enumerate(top_k_idx)}
    colour_arr = np.array([topk_set.get(t, -1) for t in y_true])
    for k, cls_idx in enumerate(top_k_idx):
        mask = y_true == cls_idx
        if mask.sum() == 0:
            continue
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=[cmap_pts(k)], s=14, alpha=0.65, zorder=2,
                   label=top_k_names[k])

    # Cluster ellipses (1-sigma) and centroids
    cluster_cmap = plt.cm.get_cmap('Set1', n_clusters)
    for c in range(n_clusters):
        cx, cy, w, h, angle = ellipses[c]
        ellipse = mpatches.Ellipse(
            (cx, cy), width=w, height=h, angle=angle,
            edgecolor=cluster_cmap(c), facecolor='none',
            linewidth=2, linestyle='--', zorder=3, alpha=0.8
        )
        ax.add_patch(ellipse)
        # Centroid star
        ax.scatter(cx, cy, marker='*', s=280, color=cluster_cmap(c),
                   edgecolors='black', linewidths=0.6, zorder=5)
        # Label: dominant true class for this cluster
        dom = summary_df.loc[summary_df['Cluster'] == c, 'Dominant_true'].values[0]
        dom_pct = summary_df.loc[summary_df['Cluster'] == c, 'Dominant_pct'].values[0]
        ax.annotate(f'C{c}: {dom}\n({dom_pct:.0f}%)',
                    (cx, cy), textcoords='offset points', xytext=(6, 6),
                    fontsize=7.5, color=cluster_cmap(c), fontweight='bold', zorder=6)

    # Overlap annotations
    for (a, b) in overlapping_pairs:
        cx_a, cy_a = ellipses[a][0], ellipses[a][1]
        cx_b, cy_b = ellipses[b][0], ellipses[b][1]
        mid_x = (cx_a + cx_b) / 2
        mid_y = (cy_a + cy_b) / 2
        ax.annotate('', xy=(cx_b, cy_b), xytext=(cx_a, cy_a),
                    arrowprops=dict(arrowstyle='<->', color=_RED,
                                   lw=1.4, linestyle='dotted'), zorder=4)
        ax.text(mid_x, mid_y, '⇔ overlap', fontsize=7,
                color=_RED, ha='center', va='bottom', zorder=6)

    # Title & labels
    n_overlap = len(overlapping_pairs)
    ax.set_title(
        f'Prediction Space Clusters — Top-{top_k} Classes  |  {model_name}\n'
        f'{reduction_label}  |  {n_clusters} K-Means clusters  |  '
        f'Silhouette = {sil:.3f}  |  '
        f'{n_overlap} overlapping cluster pair(s)',
        fontsize=10, color=_DARK, pad=10
    )
    ax.set_xlabel('Component 1', fontsize=11)
    ax.set_ylabel('Component 2', fontsize=11)
    ax.grid(alpha=0.2)

    # Legend: top-K class colours (capped at 15 entries for readability)
    legend_elements = [
        Line2D([0],[0], marker='o', color='w', markerfacecolor=cmap_pts(k),
               markersize=7, label=top_k_names[k])
        for k in range(min(top_k, 15))
    ]
    if top_k > 15:
        legend_elements.append(
            Line2D([0],[0], marker='o', color='w', markerfacecolor='#DDDDDD',
                   markersize=7, label=f'… +{top_k-15} more')
        )
    legend_elements.append(
        Line2D([0],[0], marker='*', color='w', markerfacecolor='black',
               markersize=10, label='Cluster centroid (★)')
    )
    ax.legend(handles=legend_elements, fontsize=7, loc='upper right',
              title='True class (top-K)', title_fontsize=8,
              framealpha=0.85, ncol=2)

    plt.tight_layout()
    png_path = os.path.join(END_DIR, f'{model_name}_cluster_{tag}.png')
    plt.savefig(png_path, dpi=150, bbox_inches='tight')
    plt.close()

    print(f'  ✅ Cluster {tag:6s} PNG     : {os.path.basename(png_path)}')
    print(f'     Silhouette={sil:.3f}  |  '
          f'{n_overlap} overlap pair(s)  |  '
          f'Top-{top_k} classes: {", ".join(top_k_names[:5])}'
          + (f' … +{top_k-5} more' if top_k > 5 else ''))
    print(f'  ✅ Cluster {tag:6s} CSV     : {os.path.basename(csv_path)}')
    return summary_df


# ══════════════════════════════════════════════════════════════════════════
# 5.  MASTER RUNNER
# ══════════════════════════════════════════════════════════════════════════

def analyse_model(slot, model_name):
    """
    Full analysis pipeline for one model:
      confusion table → ROC curves → cluster maps (top-1, top-5, top-20)
    All outputs saved to results/end_point/. Existing files are overwritten.
    """
    sep = '=' * 64
    print(f'\n{sep}')
    print(f'  {model_name}')
    print(sep)

    y_true, y_pred, y_proba = load_proba_and_labels(slot)
    print(f'  Loaded: {len(y_true):,} samples  |  '
          f'{y_proba.shape[1]} classes  |  '
          f'Accuracy = {(y_true==y_pred).mean()*100:.2f}%')

    print('\n  ── Confusion table ──────────────────────────────────────')
    plot_confusion_table(y_true, y_pred, LABEL_NAMES, model_name, top_n=25)

    print('\n  ── ROC curves ───────────────────────────────────────────')
    plot_roc_curves(y_true, y_proba, LABEL_NAMES, model_name, top_n_classes=10)

    print('\n  ── Cluster analysis ─────────────────────────────────────')
    for k in [1, 5, 20]:
        plot_cluster_analysis(y_true, y_proba, LABEL_NAMES, model_name, top_k=k)

    print(f'\n  ✅ {model_name} complete. All outputs → {END_DIR}')


print('✅ All functions defined.')


✅ All functions defined.


In [9]:
# ── Cell 5: Model registry — 154-class experiment ────────────────────────
P = PROJECT_DIR
R = RESULTS_DIR

MODELS_154 = [
    dict(
        name       = 'InceptionTime_154',
        npy_path   = f'{R}/inc_154/inc_154_proba_test.npy',
        y_true_src = 'npz',
        y_src_path = f'{P}/data/npz/ASL_154_test_cif.npz',
    ),
    dict(
        name       = 'RandomForest_154',
        npy_path   = f'{R}/rf_154/rf_154_proba_test.npy',
        y_true_src = 'csv',
        y_src_path = f'{P}/data/csv/ASL_154_test_rf.csv',
    ),
    dict(
        name       = 'LogisticRegression_154',
        npy_path   = f'{R}/lr_154/lr_154_proba_test.npy',
        y_true_src = 'csv',
        y_src_path = f'{P}/data/csv/ASL_154_test_rf.csv',
    ),
    dict(
        name       = 'CIF_50trees_154',
        npy_path   = f'{R}/cif_154/cif_154_proba_test.npy',
        y_true_src = 'npz',
        y_src_path = f'{P}/data/npz/ASL_154_test_cif.npz',
    ),
]

print('Registry — 154-class models:')
print(f'{"Model":<30}  {"Proba .npy":<10}  {"y_true src":<10}')
print('-' * 62)
for s in MODELS_154:
    npy_ok = '\u2705' if os.path.exists(s['npy_path']) else '\u274c MISSING'
    src_ok = '\u2705' if os.path.exists(s['y_src_path']) else '\u274c MISSING'
    print(f'{s["name"]:<30}  npy:{npy_ok}  src:{src_ok}')


Registry — 154-class models:
Model                           Proba .npy  y_true src
--------------------------------------------------------------
InceptionTime_154               npy:✅  src:✅
RandomForest_154                npy:✅  src:✅
LogisticRegression_154          npy:✅  src:✅
CIF_50trees_154                 npy:✅  src:✅


In [10]:
# ── Cell 6: Run all models ────────────────────────────────────────────────
# Each model produces 8 files in results/end_point/:
#   _confusion_table.png, _confusion_pairs.csv
#   _roc_macro.png, _roc_per_class.png
#   _cluster_top1.png,  _cluster_top1_summary.csv
#   _cluster_top5.png,  _cluster_top5_summary.csv
#   _cluster_top20.png, _cluster_top20_summary.csv
#
# Existing files are overwritten silently.

for slot in MODELS_154:
    analyse_model(slot, model_name=slot['name'])

print('\n' + '='*64)
print('  ALL MODELS COMPLETE')
print(f'  Outputs in: {END_DIR}')
print('='*64)



  InceptionTime_154
  Loaded: 309 samples  |  154 classes  |  Accuracy = 35.60%

  ── Confusion table ──────────────────────────────────────
  ✅ Confusion CSV       : InceptionTime_154_confusion_pairs.csv
  ✅ Confusion table PNG : InceptionTime_154_confusion_table.png

  ── ROC curves ───────────────────────────────────────────
  ✅ Macro ROC PNG       : InceptionTime_154_roc_macro.png  (AUC=0.9274)
  ✅ Per-class ROC PNG   : InceptionTime_154_roc_per_class.png

  ── Cluster analysis ─────────────────────────────────────
  ✅ Cluster top1   PNG     : InceptionTime_154_cluster_top1.png
     Silhouette=0.941  |  0 overlap pair(s)  |  Top-1 classes: yes
  ✅ Cluster top1   CSV     : InceptionTime_154_cluster_top1_summary.csv
  ✅ Cluster top5   PNG     : InceptionTime_154_cluster_top5.png
     Silhouette=0.807  |  0 overlap pair(s)  |  Top-5 classes: yes, like, delay, take, hot
  ✅ Cluster top5   CSV     : InceptionTime_154_cluster_top5_summary.csv
  ✅ Cluster top20  PNG     : InceptionTime_1

---
## Run a single model individually
Use the cell below to re-run just one model without re-running everything.


In [11]:
# ── Optional: single-model re-run ────────────────────────────────────────
# Change the index to target a different model:
#   0 = InceptionTime, 1 = RandomForest, 2 = LogisticRegression, 3 = CIF

TARGET = MODELS_154[3]   # ← CIF (50 trees)
analyse_model(TARGET, model_name=TARGET['name'])



  CIF_50trees_154
  Loaded: 309 samples  |  154 classes  |  Accuracy = 5.83%

  ── Confusion table ──────────────────────────────────────
  ✅ Confusion CSV       : CIF_50trees_154_confusion_pairs.csv
  ✅ Confusion table PNG : CIF_50trees_154_confusion_table.png

  ── ROC curves ───────────────────────────────────────────
  ✅ Macro ROC PNG       : CIF_50trees_154_roc_macro.png  (AUC=0.5855)
  ✅ Per-class ROC PNG   : CIF_50trees_154_roc_per_class.png

  ── Cluster analysis ─────────────────────────────────────
  ✅ Cluster top1   PNG     : CIF_50trees_154_cluster_top1.png
     Silhouette=0.942  |  0 overlap pair(s)  |  Top-1 classes: before
  ✅ Cluster top1   CSV     : CIF_50trees_154_cluster_top1_summary.csv
  ✅ Cluster top5   PNG     : CIF_50trees_154_cluster_top5.png
     Silhouette=0.902  |  0 overlap pair(s)  |  Top-5 classes: before, no, champion, change, play
  ✅ Cluster top5   CSV     : CIF_50trees_154_cluster_top5_summary.csv
  ✅ Cluster top20  PNG     : CIF_50trees_154_cluster_